[![View on GitHub](https://img.shields.io/badge/View_on-GitHub-181717?logo=github&logoColor=white)](https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/HY-World-2.0_Colab.ipynb)


# 🌍 HY-World 2.0 — Multi-Modal World Reconstruction (Tencent-Hunyuan, Apache-2.0 + Rerun viewer)

A Colab port of the [Tencent-Hunyuan/HY-World-2.0](https://huggingface.co/tencent/HY-World-2.0) **WorldMirror 2.0** world-reconstruction model. Upload a handful of images (or a short video) and the model predicts:

- **3D scene point cloud** with per-view colors
- **3D Gaussian Splatting (3DGS) `.ply`** you can open in [SuperSplat](https://supersplat.xyz), PlayCanvas, or any gsplat.js viewer
- **GLB mesh** of the scene (point cloud **or** triangulated surface)
- **Per-view depth maps** (`.png` + `.npy`)
- **Per-view surface normals** (`.png`)
- **Camera poses** (`.json` with c2w 4×4 matrices + 3×3 intrinsics)
- **Rerun `.rrd` recordings** for desktop viewing
- **In-browser 3D + splat visualisation** via [`gradio_rerun`](https://pypi.org/project/gradio_rerun/)

Powered by the [DINOv2 ViT-L/14](https://github.com/facebookresearch/dinov2) backbone (frozen) + a 24-block Visual Geometry Transformer trained by Tencent-Hunyuan. This notebook uses the small **WorldMirror 2.0** variant (~1.2 B params, 5 GB safetensors). The two bigger variants in the same repo (**WorldStereo ~17 B** and **HY-Pano ~80 B**) are not runnable on a single 24 GB Colab GPU.

---

## ⚠️ License — Territory Restriction + MAU Cap

This notebook bundles the **HY-WorldMirror-2.0** weights which are released under the **Tencent HY-WORLD 2.0 Community License Agreement** (see `License.txt` in the model repo).

- **Territory**: this license **does not apply in the European Union, the United Kingdom, or South Korea**. If you are located in any of those jurisdictions, you are **not licensed** to use these weights and should not run this notebook there. Continuing past this cell is your acceptance of the license.
- **Commercial use cap**: if your product or service exceeds **1 million monthly active users (MAU)**, you must request a separate commercial license from `hunyuan3d@tencent.com`.
- **Notice file**: any distribution of the model or its outputs must include the verbatim notice from §3(d) of the Agreement. This notebook includes that notice text in `STEP 2` so you can copy it into your distribution.
- **Output ownership**: Tencent claims no rights in the Outputs you generate. You are solely responsible for Outputs and their subsequent uses.

The vendored `hyworldmirror/` Python package is the official code from [Tencent-Hunyuan/HY-World-2.0](https://github.com/Tencent-Hunyuan/HY-World-2.0) and is **Apache-2.0 licensed**. Only the **weights** carry the restricted license above.

By running this notebook you accept the Tencent HY-WORLD 2.0 Community License Agreement on behalf of yourself and your organisation.

---

## Quick start

1. **Runtime → Change runtime type → GPU** (A100 or L4 recommended; T4 may OOM during inference).
2. Run **STEP 1** — installs gsplat + PyTorch 2.4 + clones the Space repo + patches `attention.py` to use PyTorch SDPA instead of flash-attn (flash-attn wheels require torch 2.8 but gsplat only ships wheels for torch 2.4).
3. Run **STEP 2** — verifies GPU, exports the Drive cache prologue, defines the lazy model loader.
4. Run **STEP 4** — opens the Gradio UI. Upload images or a video, choose reconstruction options, click **Reconstruct 3D Scene**.
5. **STEP 6** gives a single-image quick test; **STEP 7** processes an entire folder of image sets in batch.

Estimated first-run setup: **~8 minutes** (5 GB weight download + torch re-install + gsplat wheel). Subsequent runs reuse the Drive cache and take ~30 s.

## Outputs

Everything is written to `AEI_3D_Cache/HY-World-2.0/recon_<uuid>/`:

```
scene_<selector>.glb          # mesh (or point cloud) you can open anywhere
gaussians.ply                 # 3DGS splats → SuperSplat / PlayCanvas
depth_000.png … depth_NNN.png # colormap depth for each input frame
depth_000.npy … depth_NNN.npy # raw metric depth (HxW float32)
normal_000.png … normal_NNN.png
camera_poses.json             # {c2w: [N,4,4], K: [N,3,3]}
reconstruction.rrd            # Rerun recording (3D + cameras + normals)
gaussians.rrd                 # Rerun recording (Gaussian splat cloud)
```

## Companion notebooks

- **MapAnything_Colab** — Apache-2.0, single-image → 3D in ~10 s
- **NoPoSplat_Colab** — MIT, 2-3 photos → 3DGS in ~10 s
- **Pi3X_Colab** — video → 3DGS, BSD-3 code / CC BY-NC-4.0 weights
- **WildGaussianSplatting_Colab** — image folder / video → 3DGS in 5–15 min, INRIA non-commercial
- **SplatTransform_Colab** — 3DGS format converter + voxel collision mesh
- **SkinTokens_Colab** — mesh → rig (TokenRig, MIT)
- **TripoSplat_Colab** — text/image → 3DGS via TripoSR + gsplat
- **TextureMapPrep_Colab** — seamless PBR maps for game assets


In [ ]:
#@title STEP 1 — Install gsplat, PyTorch 2.4, clone Space + HY-World-2.0, patch attention.py
"""
• Detects GPU + CUDA, installs PyTorch 2.4.0 + cu124 (matches the only
  gsplat 1.5.3 wheel that exists — `pt24cu124`).
• Installs the pre-built gsplat 1.5.3 wheel (CUDA rasterizer) and all of
  the other inference-time deps from the upstream Space requirements.
• Pins Gradio 6.9.0 + gradio_rerun 0.32.2 (Gradio 5.49.1 has no compatible
  gradio_rerun release; gradio_rerun requires gradio>=6.  We pin to 6.9.0
  because that is the latest stable Gradio 6.x actually published to PyPI
  — the Space's `gradio==6.12.0` does not exist).
• git-clones the Space `hyworldmirror/` Python package (Apache-2.0) into
  `/content/hyworldmirror/`. This is the only place to get the model code
  — there is no PyPI release.
• git-clones the [Tencent-Hunyuan/HY-World-2.0](https://github.com/Tencent-Hunyuan/HY-World-2.0)
  reference repo for the GLB-mesh assembly helpers and the README.
• Patches the vendored `attention.py` to **drop flash-attn** and route
  attention through PyTorch SDPA instead. Reason: flash-attn pre-built
  wheels are pinned to torch 2.8 while gsplat wheels are pinned to torch
  2.4, so they cannot coexist. SDPA with bf16 is performant enough for
  a 24-block ViT-L on a 24 GB GPU.  The original `attention.py` is
  backed up to `attention.py.bak` so you can restore it after a future
  gsplat wheel adds torch 2.8 support.
• Pre-downloads the 5 GB WorldMirror 2.0 safetensors into the Drive cache
  so subsequent runs do not re-download.
"""
import os, sys, time, json, subprocess, shutil, pathlib, traceback, glob, importlib
import requests

print('='*72)
print('HY-World 2.0 / WorldMirror — Install + Setup')
print('='*72)
try:
    import torch
    print(f'  torch        : {torch.__version__}  (CUDA {torch.version.cuda or "none"})')
    print(f'  CUDA avail   : {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            p = torch.cuda.get_device_properties(i)
            print(f'  GPU {i}        : {p.name}  ({p.total_memory / (1024**3):.1f} GB)')
    else:
        print('  WARNING: no GPU detected — inference will fall back to CPU (very slow)')
except ImportError:
    torch = None
    print('  torch not yet installed (will be installed below)')
print()

CONNECT_GOOGLE_DRIVE = True  #@param {type:'boolean'}
if CONNECT_GOOGLE_DRIVE:
    drive_root = pathlib.Path('/content/drive/MyDrive/AEI_3D_Cache/HY-World-2.0')
    drive_root.mkdir(parents=True, exist_ok=True)
    print(f'  Drive cache  : {drive_root}')
    os.environ['HF_HOME'] = str(drive_root / 'huggingface')
    os.environ['HUGGINGFACE_HUB_CACHE'] = str(drive_root / 'huggingface')
else:
    drive_root = pathlib.Path('/content/_hyworld_cache')
    drive_root.mkdir(parents=True, exist_ok=True)
    os.environ['HF_HOME'] = str(drive_root / 'huggingface')
    print(f'  Local cache  : {drive_root}  (lost on runtime disconnect)')

# Where the vendored model package lives on the Colab VM
WORK_ROOT   = pathlib.Path('/content/hyworld_work')
WORK_ROOT.mkdir(parents=True, exist_ok=True)
REPO_DIR    = WORK_ROOT / 'hyworldmirror_repo'   # the Space repo (has hyworldmirror/ + app.py)
MODEL_DIR   = WORK_ROOT / 'hyworldmirror_pkg'    # the cloned HY-World-2.0 reference repo
SPACES_REPO = 'prithivMLmods/HY-World-2.0-Demo'
HY_REPO_URL = 'https://github.com/Tencent-Hunyuan/HY-World-2.0.git'

t_total = time.time()

# 1. PyTorch 2.4.0 + cu124 (only torch that has a gsplat 1.5.3 wheel) ────
TORCH_TARGET = '2.4.0'
NEED_TORCH = (
    torch is None
    or not torch.cuda.is_available()
    or not torch.__version__.startswith(TORCH_TARGET)
)
if NEED_TORCH:
    print(f'\n[1/8] Installing PyTorch {TORCH_TARGET} + cu124 ...')
    t0 = time.time()
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade',
         f'torch=={TORCH_TARGET}', 'torchvision==0.19.0', 'torchaudio==2.4.0',
         '--index-url', 'https://download.pytorch.org/whl/cu124'],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        print('  ERROR:', r.stderr[-1500:])
        raise RuntimeError('PyTorch install failed')
    print(f'  PyTorch installed in {time.time()-t0:.1f}s')
    # Re-import so the rest of the cell sees the new torch
    importlib.reload(importlib.import_module('torch'))
    import torch
    print(f'  torch now    : {torch.__version__}  (CUDA {torch.version.cuda})')
else:
    print(f'\n[1/8] torch {torch.__version__} already matches {TORCH_TARGET} — skipping install')

# 2. gsplat 1.5.3 (only wheel for torch 2.4 + cu124) ──────────────────────
print('\n[2/8] Installing gsplat 1.5.3 (pt24cu124 wheel) ...')
t0 = time.time()
GSPLAT_WHEEL = (
    'https://github.com/nerfstudio-project/gsplat/releases/download/'
    'v1.5.3/gsplat-1.5.3+pt24cu124-cp310-cp310-linux_x86_64.whl'
)
r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', GSPLAT_WHEEL],
    capture_output=True, text=True,
)
if r.returncode != 0:
    print('  ERROR:', r.stderr[-1500:])
    raise RuntimeError('gsplat install failed')
print(f'  gsplat installed in {time.time()-t0:.1f}s')

# 3. Inference-time deps from the Space requirements.txt ────────────────
print('\n[3/8] Installing inference-time deps ...')
t0 = time.time()
# Note: we deliberately OMIT flash-attn and torch==2.8.0 — they conflict
# with gsplat 1.5.3 / torch 2.4.  SDPA on bf16 covers us.
EXTRA_PKGS = [
    'gradio==6.9.0',
    'gradio_rerun==0.32.2',
    'rerun-sdk==0.32.2',
    'huggingface_hub',
    'safetensors',
    'opencv-python-headless',
    'pillow_heif',
    'moviepy==1.0.3',
    'trimesh>=4.5,<5.0',
    'open3d==0.18.0',
    'plyfile',
    'numpy>=2.0,<2.3',  # rerun-sdk 0.32.2 requires numpy>=2; upstream code is fine on 2.x
    'scipy',
    'matplotlib',
    'tqdm',
    'omegaconf',
    'einops',
    'torchmetrics',
    'jaxtyping',
    'typeguard',
    'colorspacious',
    'onnxruntime',
    'kernels',
    'uniception',
    'requests',
    'fastapi',
    'pydantic',
    'termcolor',
    'jax',
]
r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet'] + EXTRA_PKGS,
    capture_output=True, text=True,
)
if r.returncode != 0:
    print('  ERROR:', r.stderr[-2000:])
    raise RuntimeError('Inference deps install failed')
print(f'  Inference deps installed in {time.time()-t0:.1f}s')

# 4. Clone the Space repo (gives us the hyworldmirror/ package + app.py) ─
print(f'\n[4/8] Cloning the {SPACES_REPO} Space repo (vendored hyworldmirror/) ...')
t0 = time.time()
if (REPO_DIR / 'hyworldmirror' / '__init__.py').exists():
    print('  Already cloned — pulling latest')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--quiet', '--ff-only'],
                   capture_output=True)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR, ignore_errors=True)
    r = subprocess.run(
        ['git', 'clone', '--quiet', '--depth=1',
         f'https://huggingface.co/spaces/{SPACES_REPO}', str(REPO_DIR)],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        print('  ERROR:', r.stderr[-1500:])
        raise RuntimeError('Space clone failed')
print(f'  Space repo cloned in {time.time()-t0:.1f}s')

# Make the vendored hyworldmirror importable from anywhere
SITE_PKG = pathlib.Path(sys.executable).parent.parent / 'lib' / f'python{sys.version_info.major}.{sys.version_info.minor}' / 'site-packages'
HWM_LINK = SITE_PKG / 'hyworldmirror'
if HWM_LINK.exists() or HWM_LINK.is_symlink():
    if HWM_LINK.is_symlink() or HWM_LINK.is_file():
        HWM_LINK.unlink()
    else:
        shutil.rmtree(HWM_LINK, ignore_errors=True)
HWM_LINK.symlink_to(REPO_DIR / 'hyworldmirror', target_is_directory=True)
print(f'  Linked hyworldmirror → {HWM_LINK}')

# 5. Clone the upstream Tencent-Hunyuan/HY-World-2.0 reference repo ──────
print(f'\n[5/8] Cloning {HY_REPO_URL} (for README + License + examples) ...')
t0 = time.time()
if (MODEL_DIR / 'README.md').exists():
    print('  Already cloned — pulling latest')
    subprocess.run(['git', '-C', str(MODEL_DIR), 'pull', '--quiet', '--ff-only'],
                   capture_output=True)
else:
    if MODEL_DIR.exists():
        shutil.rmtree(MODEL_DIR, ignore_errors=True)
    r = subprocess.run(
        ['git', 'clone', '--quiet', '--depth=1', HY_REPO_URL, str(MODEL_DIR)],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        print('  WARN: clone failed (non-fatal):', r.stderr[-500:])
print(f'  Reference repo cloned in {time.time()-t0:.1f}s')

# 6. Patch attention.py to drop flash-attn and use SDPA ──────────────────
print('\n[6/8] Patching hyworldmirror/models/layers/attention.py → PyTorch SDPA ...')
t0 = time.time()
ATTN_PATH = REPO_DIR / 'hyworldmirror' / 'models' / 'layers' / 'attention.py'
ATTN_BACKUP = REPO_DIR / 'hyworldmirror' / 'models' / 'layers' / 'attention.py.bak'
attn_src = ATTN_PATH.read_text()

# Build a stub module so the `from flash_attn_interface import ...` lines
# at the top of the file succeed at import time without installing the
# actual flash-attn library.  The stub exposes flash_attn_func with the
# same signature; the patched _apply_attention below routes around it.
STUB = (
    '\n# --- AEI-Colab patch: flash-attn stub (replaced by SDPA below) ---\n'
    'class _FlashAttnStub:\n'
    '    def __getattr__(self, name):\n'
    '        return self\n'
    '    def __call__(self, *args, **kwargs):\n'
    '        raise RuntimeError("flash-attn is not installed in this Colab; SDPA is used instead")\n'
    'flash_attn_func_v3 = _FlashAttnStub()\n'
    'flash_attn_func_v2 = _FlashAttnStub()\n'
    '_USE_FLASH_ATTN_V3 = False\n'
    'try:\n'
    '    import flash_attn.flash_attn_interface  # noqa: F401\nexcept ImportError:\n'
    '    pass\n'
    '# --- end patch ---\n'
)

# Replace the `try: from flash_attn_interface import ...` block with our stub
needle = (
    'try:\n'
    '    from flash_attn_interface import flash_attn_func as flash_attn_func_v3\n'
    '    _USE_FLASH_ATTN_V3 = True\n'
    'except ImportError:\n'
    '    from flash_attn.flash_attn_interface import flash_attn_func as flash_attn_func_v2\n'
    '    _USE_FLASH_ATTN_V3 = False'
)
if needle in attn_src:
    attn_src = attn_src.replace(needle, STUB.rstrip())
else:
    print('  WARN: flash-attn import block not found verbatim — inserting stub after imports anyway')
    lines = attn_src.split('\n')
    insert_at = 0
    for i, ln in enumerate(lines[:30]):
        if ln.startswith('import') or ln.startswith('from'):
            insert_at = i + 1
    lines.insert(insert_at, STUB.rstrip())
    attn_src = '\n'.join(lines)

# Now rewrite the `_apply_attention` body to always use SDPA, regardless
# of dtype.  The original code uses `flash_attn_func` whenever q is bf16 /
# fp16; we replace that whole branch with a single `F.scaled_dot_product_attention`
# call that takes standard [B, H, N, C] tensors.
old_apply = (
    '    def _apply_attention(self, q: Tensor, k: Tensor, v: Tensor) -> Tensor:\n'
    '        if q.dtype==torch.bfloat16 or q.dtype==torch.float16:\n'
    '            if q.is_contiguous():\n'
    '                q = q.transpose(1,2)\n'
    '            else:\n'
    '                q = q.transpose(1, 2).contiguous()\n'
    '            if k.is_contiguous():\n'
    '                k = k.transpose(1, 2)\n'
    '            else:\n'
    '                k = k.transpose(1, 2).contiguous()\n'
    '            if v.is_contiguous():\n'
    '                v = v.transpose(1, 2)\n'
    '            else:\n'
    '                v = v.transpose(1, 2).contiguous()\n'
    '            if _USE_FLASH_ATTN_V3:\n'
    '                x = flash_attn_func_v3(q, k, v)\n'
    '            else:\n'
    '                x = flash_attn_func_v2(q, k, v, dropout_p=self.attn_drop.p if self.training else 0.0)\n'
    '            if x.is_contiguous():\n'
    '                x = x.transpose(1, 2)\n'
    '            else:\n'
    '                x = x.transpose(1, 2).contiguous()\n'
    '        else:\n'
    '            x = F.scaled_dot_product_attention(q, k, v, dropout_p=self.attn_drop.p if self.training else 0.0)\n'
    '        return x'
)
new_apply = (
    '    def _apply_attention(self, q: Tensor, k: Tensor, v: Tensor) -> Tensor:\n'
    '        # AEI-Colab patch: always use PyTorch SDPA.  flash-attn pre-built\n'
    '        # wheels require torch 2.8; gsplat 1.5.3 requires torch 2.4.  SDPA\n'
    '        # with bf16 autocast is the right path for a 24 GB GPU.\n'
    '        return F.scaled_dot_product_attention(\n'
    '            q, k, v,\n'
    '            dropout_p=self.attn_drop.p if self.training else 0.0,\n'
    '            is_causal=False,\n'
    '        )'
)
if old_apply in attn_src:
    attn_src = attn_src.replace(old_apply, new_apply)
    if not ATTN_BACKUP.exists():
        ATTN_BACKUP.write_text(ATTN_PATH.read_text())
    ATTN_PATH.write_text(attn_src)
    print(f'  attention.py patched in {time.time()-t0:.1f}s')
else:
    print('  ERROR: _apply_attention body not found verbatim — patch aborted')
    print('         The model will crash on first inference.  Re-run STEP 1')
    print('         or apply the SDPA patch manually.')
    raise RuntimeError('attention.py patch failed — body not found')

# 7. Pre-download the 5 GB WorldMirror 2.0 safetensors into Drive cache ─
print('\n[7/8] Pre-downloading HY-WorldMirror-2.0 weights (5 GB) into Drive cache ...')
t0 = time.time()
from huggingface_hub import snapshot_download
WEIGHTS_DIR = drive_root / 'huggingface' / 'models--tencent--HY-World-2.0' / 'snapshots'
if WEIGHTS_DIR.exists() and any(WEIGHTS_DIR.rglob('model.safetensors')):
    print('  Weights already cached on Drive — skipping download')
else:
    snapshot_download(
        repo_id='tencent/HY-World-2.0',
        allow_patterns=['HY-WorldMirror-2.0/**', 'License.txt'],
        cache_dir=str(drive_root / 'huggingface'),
        etag_timeout=30,
    )
print(f'  Weights ready in {time.time()-t0:.1f}s')

# 8. Verify the model can be imported and a small dummy forward works ───
print('\n[8/8] Smoke test — import WorldMirror and build the model (no weights) ...')
t0 = time.time()
sys.path.insert(0, str(REPO_DIR))
try:
    from hyworldmirror.models.models.worldmirror import WorldMirror
    with open(REPO_DIR / 'hyworldmirror' / 'models' / 'models' / 'worldmirror.py') as f:
        pass  # file exists
    print('  WorldMirror import path resolves OK')
    # Build model with default config
    import torch as _t
    _cfg_path = WEIGHTS_DIR / list(WEIGHTS_DIR.iterdir())[0] / 'HY-WorldMirror-2.0' / 'config.json' if WEIGHTS_DIR.exists() else None
    if _cfg_path and _cfg_path.exists():
        with open(_cfg_path) as f:
            _cfg = json.load(f)
        _cfg.pop('_target_', None)
        _m = WorldMirror(**{k: v for k, v in _cfg.items() if k in {
            'img_size','patch_size','model_size','embed_dim','depth','num_heads',
            'mlp_ratio','gs_dim','num_register_tokens','enable_cond','enable_cam',
            'enable_pts','enable_depth','enable_depth_mask','enable_norm','enable_gs',
            'enable_bf16','patch_embed','fixed_patch_embed','sampling_strategy',
            'dpt_gradient_checkpoint','condition_strategy','rope_base','normalized_rope',
            'rope_normalize_coords','rope_shift_coords','rope_jitter_coords',
            'rope_rescale_coords','sp_size',
        }})
        n_params = sum(p.numel() for p in _m.parameters())
        print(f'  WorldMirror built (no weights yet): {n_params/1e6:.1f} M params')
        del _m
    else:
        print('  Skipping model build (weights not yet on disk)')
except Exception as e:
    print(f'  WARN: model import smoke test failed: {e}')
    traceback.print_exc(limit=4)
print(f'  Smoke test took {time.time()-t0:.1f}s')

# Final summary
elapsed = time.time() - t_total
print()
print('='*72)
print(f'STEP 1 complete in {elapsed/60:.1f} min')
print('='*72)
print(f'  Drive cache        : {drive_root}')
print(f'  Work dir           : {WORK_ROOT}')
print(f'  Vendored pkg       : {REPO_DIR / "hyworldmirror"}  (Apache-2.0)')
print(f'  Reference repo     : {MODEL_DIR}')
print(f'  Patched attention  : {ATTN_PATH}  (backup at {ATTN_BACKUP})')
print()
print('Next: run STEP 2 (cache + imports + lazy model loader).')


In [ ]:
#@title STEP 2 — Imports, Drive cache, lazy model loader, license notice
"""
• Verifies GPU + Python + PyTorch + gsplat + gradio_rerun versions
• Sets the persistent Drive cache prologue (HF_HOME, HUGGINGFACE_HUB_CACHE)
• Defines `load_worldmirror()` — the lazy loader that downloads weights on
  first call and returns `(model, device, model_dir)` afterwards
• Defines the **verbatim** §3(d) Notice text required by the Tencent
  HY-WORLD 2.0 Community License for downstream distributions
• Wires in a few compatibility shims (e.g. scipy.spatial.transform.Rotation
  used in visual_util.py — we make sure it's importable; trimesh 4.x vs 3.x
  face ordering; etc.)
"""
import os, sys, time, json, gc, uuid, glob, shutil, subprocess, importlib
import pathlib

print('='*72)
print('HY-World 2.0 / WorldMirror — Imports + cache + lazy loader')
print('='*72)

# --- Version / dep checks ──────────────────────────────────────────────
import torch
assert torch.__version__.startswith('2.4'), (
    f'Expected torch 2.4.x for gsplat 1.5.3 wheel — got {torch.__version__}.\n'
    'Re-run STEP 1 to fix this.'
)
import gradio as gr
print(f'  torch        : {torch.__version__}  (CUDA {torch.version.cuda})')
print(f'  gradio       : {gr.__version__}')

try:
    import gsplat
    print(f'  gsplat       : {gsplat.__version__}')
except Exception as e:
    print(f'  gsplat       : MISSING ({e})')
    raise

try:
    import rerun as rr
    print(f'  rerun-sdk    : {rr.__version__}')
except Exception as e:
    print(f'  rerun-sdk    : MISSING ({e})')
    raise

try:
    from gradio_rerun import Rerun
    print(f'  gradio_rerun : OK')
except Exception as e:
    print(f'  gradio_rerun : MISSING ({e})')
    raise

import numpy as np
import cv2
import trimesh
import requests
from PIL import Image
from tqdm.auto import tqdm
print(f'  numpy        : {np.__version__}')
print(f'  opencv       : {cv2.__version__}')
print(f'  trimesh      : {trimesh.__version__}')
print()

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}        : {p.name}  ({p.total_memory / (1024**3):.1f} GB)')
else:
    print('  WARNING: no GPU detected — WorldMirror inference will be painfully slow')

# --- Drive cache prologue (re-affirm — also useful if user re-runs cell) ─
CONNECT_GOOGLE_DRIVE = True  #@param {type:'boolean'}
if CONNECT_GOOGLE_DRIVE:
    drive_root = pathlib.Path('/content/drive/MyDrive/AEI_3D_Cache/HY-World-2.0')
    drive_root.mkdir(parents=True, exist_ok=True)
    HF_HOME = drive_root / 'huggingface'
    HF_HOME.mkdir(parents=True, exist_ok=True)
    os.environ['HF_HOME'] = str(HF_HOME)
    os.environ['HUGGINGFACE_HUB_CACHE'] = str(HF_HOME)
    CACHE_ROOT = drive_root
else:
    drive_root = pathlib.Path('/content/_hyworld_cache')
    drive_root.mkdir(parents=True, exist_ok=True)
    HF_HOME = drive_root / 'huggingface'
    HF_HOME.mkdir(parents=True, exist_ok=True)
    os.environ['HF_HOME'] = str(HF_HOME)
    os.environ['HUGGINGFACE_HUB_CACHE'] = str(HF_HOME)
    CACHE_ROOT = drive_root

OUT_DIR = drive_root / 'hyworld_out'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'\n  Drive cache   : {drive_root}')
print(f'  HF_HOME       : {HF_HOME}')
print(f'  Out dir       : {OUT_DIR}')

# Re-assert the work root + symlink from STEP 1
WORK_ROOT   = pathlib.Path('/content/hyworld_work')
WORK_ROOT.mkdir(parents=True, exist_ok=True)
REPO_DIR    = WORK_ROOT / 'hyworldmirror_repo'
MODEL_DIR   = WORK_ROOT / 'hyworldmirror_pkg'
sys.path.insert(0, str(REPO_DIR))

assert (REPO_DIR / 'hyworldmirror' / 'models' / 'models' / 'worldmirror.py').exists(), (
    f'hyworldmirror package not found at {REPO_DIR}.  Re-run STEP 1.'
)
print(f'  Vendored pkg  : {REPO_DIR / "hyworldmirror"}')

# --- Memory allocator config (matches the Space) ───────────────────────
os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')

# --- License notice (verbatim §3(d)) — copy/paste into your distribution ─
LICENSE_NOTICE = (
    'Tencent HY-WORLD 2.0 is licensed under the Tencent HY-WORLD 2.0 Community '
    'License Agreement, Copyright (c) 2026 Tencent. All Rights Reserved. '
    'The trademark rights of "Tencent HY" are owned by Tencent or its affiliate.'
)
print(f'\n  License notice (§3(d)) :')
print(f'    {LICENSE_NOTICE}\n')

# --- Locate the cached weights snapshot ────────────────────────────────
def _find_model_snapshot():
    """Walk the Drive cache and return the path to HY-WorldMirror-2.0/.

    Returns None if the weights haven't been downloaded yet.
    """
    candidates = list((HF_HOME / 'models--tencent--HY-World-2.0' / 'snapshots').glob('*/HY-WorldMirror-2.0'))
    if not candidates:
        # Fall back to a fresh snapshot_download with a quiet print
        from huggingface_hub import snapshot_download
        snap = snapshot_download(
            repo_id='tencent/HY-World-2.0',
            allow_patterns=['HY-WorldMirror-2.0/**', 'License.txt'],
            cache_dir=str(HF_HOME),
        )
        return pathlib.Path(snap) / 'HY-WorldMirror-2.0'
    return candidates[0]

# --- Lazy model loader (singleton) ─────────────────────────────────────
_MODEL = None
_DEVICE = None
_MODEL_DIR = None

def load_worldmirror(verbose=True):
    """Load the WorldMirror 2.0 model from the Drive cache (singleton).

    Returns: (model, device, model_dir)
    """
    global _MODEL, _DEVICE, _MODEL_DIR
    if _MODEL is not None:
        return _MODEL, _DEVICE, _MODEL_DIR

    from omegaconf import OmegaConf
    from safetensors.torch import load_file as load_safetensors
    from hyworldmirror.models.models.worldmirror import WorldMirror

    if verbose:
        print(f'  Lazy-loading WorldMirror 2.0 ...')
    _DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if torch.cuda.is_available() and verbose:
        print(f'  GPU: {torch.cuda.get_device_name(torch.cuda.current_device())}')

    _MODEL_DIR = _find_model_snapshot()
    if verbose:
        print(f'  Model dir: {_MODEL_DIR}')

    yaml_path = _MODEL_DIR / 'config.yaml'
    json_path = _MODEL_DIR / 'config.json'

    if yaml_path.exists():
        cfg = OmegaConf.load(yaml_path)
        if hasattr(cfg, 'wrapper') and hasattr(cfg.wrapper, 'model'):
            model_cfg = cfg.wrapper.model
        elif hasattr(cfg, 'model'):
            model_cfg = cfg.model
        else:
            model_cfg = cfg
        model_cfg = OmegaConf.to_container(model_cfg, resolve=True)
        model_cfg.pop('_target_', None)
    elif json_path.exists():
        with open(json_path) as f:
            model_cfg = json.load(f)
    else:
        raise FileNotFoundError(f'No config.yaml / config.json in {_MODEL_DIR}')

    _MODEL = WorldMirror(**model_cfg).to(_DEVICE)
    safetensors_path = _MODEL_DIR / 'model.safetensors'
    if safetensors_path.exists():
        state = load_safetensors(str(safetensors_path))
        current = _MODEL.state_dict()
        matched = 0
        for key in current:
            if key in state and current[key].shape == state[key].shape:
                current[key] = state[key]
                matched += 1
        _MODEL.load_state_dict(current, strict=True)
        del state
        if verbose:
            print(f'  Loaded {matched}/{len(current)} keys from safetensors')
    _MODEL.eval()
    for p in _MODEL.parameters():
        p.requires_grad = False
    if verbose:
        n_params = sum(p.numel() for p in _MODEL.parameters())
        print(f'  WorldMirror ready: {n_params/1e6:.1f} M params, dtype={next(_MODEL.parameters()).dtype}')
    return _MODEL, _DEVICE, _MODEL_DIR


def free_worldmirror():
    """Unload the model and free GPU memory."""
    global _MODEL, _DEVICE, _MODEL_DIR
    if _MODEL is not None:
        del _MODEL
        _MODEL = None
    _DEVICE = None
    _MODEL_DIR = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# --- Small smoke test: import the inference utils the UI will need ─────
try:
    from hyworldmirror.utils.inference_utils import (
        prepare_images_to_tensor,
        compute_adaptive_target_size,
        compute_sky_mask,
        compute_filter_mask,
        _voxel_prune_gaussians,
        depth_to_world_coords_points,
    )
    from hyworldmirror.utils.visual_util import convert_predictions_to_glb_scene
    from hyworldmirror.utils.save_utils import save_camera_params, save_gs_ply
    print('  hyworldmirror.utils imports OK')
except Exception as e:
    print(f'  WARN: hyworldmirror.utils import failed: {e}')
    traceback.print_exc(limit=4)

# Quick probe: import torch.hub utilities we will need for DINOv2 weights
try:
    torch.hub.list = lambda *a, **kw: []  # no-op for the dry-run
    print(f'  DINOv2 loader   : torch.hub (will fetch on first inference)')
except Exception:
    pass

print()
print('STEP 2 complete — model loader is ready.  Run STEP 4 to open the Gradio UI.')


In [ ]:
#@title STEP 3 — Core helpers (frame extraction, mask computation, save_gs_ply, sky onnx)
"""
• `extract_video_frames()` — decodes an uploaded video to a fresh folder
  of PNGs using OpenCV with the same fps-based interval as the Space.
• `process_uploaded_files()` — accepts a list of `gr.File` paths and
  returns `(target_dir, sorted_image_paths)`. Handles `.mp4/.avi/.mov/.mkv/
  .webm/.flv/.wmv` (video), `.heic/.heif` (Apple HEIF), and all common
  image extensions.
• `render_depth_colormap()` / `render_normal_colormap()` — for the
  per-view depth + normal PNGs.
• `_to_numpy()` / `_to_cpu_f32()` — small tensor→numpy shims the
  hyworldmirror code uses everywhere.
• `save_gs_ply_aes()` — AEI-friendly PLY writer that converts the
  model's post-activation values (exp'd scales, sigmoid'd opacities)
  back to the 3DGS convention: LOG scale + LOGIT opacity, matching
  what SuperSplat, PlayCanvas, and gsplat.js expect.
• `load_skyseg_onnx()` — pre-downloads the JianyuanWang/skyseg ONNX
  model (the one the upstream `compute_sky_mask()` actually uses) into
  `/content/hyworld_work/skyseg/`.  Returns the directory path; the
  inference code `chdir`s into it when calling `compute_sky_mask()`
  (which looks for `skyseg.onnx` in CWD) and chdirs back.
• `free_cuda()` / `log_inference_time()` — small utilities used in STEP 7
  to keep batch loops stable.
"""
import os, sys, time, json, gc, uuid, glob, shutil, subprocess, traceback
import pathlib
import cv2
import numpy as np
from PIL import Image
from datetime import datetime
from glob import glob
from concurrent.futures import ThreadPoolExecutor

# --- Constants we re-use across the notebook ───────────────────────────
VIDEO_EXTS   = {'.mp4', '.avi', '.mov', '.mkv', '.webm', '.flv', '.wmv'}
HEIF_EXTS    = {'.heic', '.heif'}
IMAGE_EXTS   = {'.png', '.jpg', '.jpeg', '.webp', '.bmp', '.tiff'}

UPLOAD_DIR   = pathlib.Path('/content/hyworld_uploads')
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

# --- Tensor ↔ numpy shims ──────────────────────────────────────────────
def _to_numpy(t):
    """torch.Tensor | np.ndarray | list → np.ndarray (cpu, float32 if numeric)."""
    if isinstance(t, torch.Tensor):
        return t.detach().cpu().numpy()
    if isinstance(t, np.ndarray):
        return t
    return np.asarray(t)

def _to_cpu_f32(t):
    return _to_numpy(t).astype(np.float32) if np.issubdtype(_to_numpy(t).dtype, np.floating) else _to_numpy(t)

# --- Frame extraction (mirrors the Space logic) ────────────────────────
def extract_video_frames(video_path: str, time_interval: float = 1.0, out_dir: pathlib.Path = None) -> list:
    """Extract frames from a video at `time_interval` seconds."""
    out_dir = pathlib.Path(out_dir) if out_dir else UPLOAD_DIR / f'video_{uuid.uuid4().hex[:8]}'
    out_dir.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    interval = max(1, int(fps * time_interval))
    frames = []
    idx = saved = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        idx += 1
        if idx % interval == 0:
            p = out_dir / f'frame_{saved:06d}.png'
            cv2.imwrite(str(p), frame)
            frames.append(str(p))
            saved += 1
    cap.release()
    return frames

def process_uploaded_files(files, time_interval: float = 1.0):
    """Accept a list of `gr.File` paths.  Returns (target_dir, sorted_image_paths)."""
    target_dir = UPLOAD_DIR / f'input_{datetime.now().strftime("%Y%m%d_%H%M%S_%f")}'
    images_dir = target_dir / 'images'
    images_dir.mkdir(parents=True, exist_ok=True)
    image_paths = []
    for f in files or []:
        src = pathlib.Path(f) if isinstance(f, str) else pathlib.Path(f.name)
        if not src.exists():
            continue
        ext = src.suffix.lower()
        base_name = src.stem
        try:
            if ext in VIDEO_EXTS:
                image_paths.extend(extract_video_frames(str(src), time_interval, images_dir))
            elif ext in HEIF_EXTS:
                try:
                    from pillow_heif import register_heif_opener
                    register_heif_opener()
                except Exception:
                    pass
                img = Image.open(str(src)).convert('RGB')
                dst = images_dir / f'{base_name}.jpg'
                img.save(dst, 'JPEG', quality=95)
                image_paths.append(str(dst))
            elif ext in IMAGE_EXTS:
                dst = images_dir / src.name
                if dst.resolve() != src.resolve():
                    shutil.copy2(str(src), str(dst))
                image_paths.append(str(dst))
            else:
                # unknown extension — try to open as image
                try:
                    img = Image.open(str(src)).convert('RGB')
                    dst = images_dir / f'{base_name}.png'
                    img.save(dst, 'PNG')
                    image_paths.append(str(dst))
                except Exception:
                    pass
        except Exception as e:
            print(f'  [process_uploaded_files] skipped {src.name}: {e}')
    return str(target_dir), sorted(image_paths)

# --- Depth / normal colormaps (used for PNG previews) ──────────────────
def render_depth_colormap(depth_map, mask=None):
    import matplotlib.pyplot as plt
    d = depth_map.copy()
    valid = (d > 0) & mask if mask is not None else (d > 0)
    if valid.sum() > 0:
        lo, hi = np.percentile(d[valid], 5), np.percentile(d[valid], 95)
        d[valid] = (d[valid] - lo) / (hi - lo + 1e-9)
    rgb = (plt.cm.turbo_r(d)[:, :, :3] * 255).astype(np.uint8)
    if mask is not None:
        rgb[~mask] = [255, 255, 255]
    return rgb

def render_normal_colormap(normal_map, mask=None):
    n = normal_map.copy()
    if mask is not None:
        n[~mask] = 0
    return ((n + 1.0) * 0.5 * 255).clip(0, 255).astype(np.uint8)

# --- 3DGS PLY writer (matches upstream save_gs_ply) ────────────────────
def save_gs_ply_aes(path, means, scales, quats, colors, opacities):
    """Write a 3DGS `.ply` that SuperSplat + PlayCanvas + gsplat.js all open.

    The 3DGS PLY convention expects pre-activation values:
      * `scale_*`  = LOG of the post-activation scale (= the raw output
        of the model's scale head before exp())
      * `opacity`  = LOGIT of the post-activation opacity (sigmoid⁻¹)
    Upstream `hyworldmirror.utils.save_utils.save_gs_ply` does this by
    writing `scales.log()` directly; we do the same here, plus we apply
    `logit()` to the post-sigmoid opacities.

    `colors` are SH DC coefficients (already in SH space from
    `sp['sh'][..., 0, :]`), so we pass them through unchanged.

    Layout:
      x, y, z                   (float32, 3)  — 3D centers
      nx, ny, nz                (float32, 3)  — always zero
      f_dc_0, f_dc_1, f_dc_2    (float32, 3)  — DC SH band
      opacity                   (float32, 1)  — logit (pre-sigmoid)
      scale_0, scale_1, scale_2 (float32, 3)  — log of post-activation
      rot_0, rot_1, rot_2, rot_3 (float32, 4) — normalized quaternion
    """
    means     = _to_numpy(means).astype(np.float32)
    scales    = _to_numpy(scales).astype(np.float32)
    quats     = _to_numpy(quats).astype(np.float32)
    colors    = _to_numpy(colors).astype(np.float32)
    opacities = _to_numpy(opacities).astype(np.float32)

    # 3DGS convention: scales are stored as LOG scale; opacities as LOGIT.
    # The model gives us post-activation (exp / sigmoid); undo them here.
    log_scales = np.log(np.clip(scales, 1e-8, None))
    opacities_clipped = np.clip(opacities, 1e-6, 1.0 - 1e-6)
    logit_opacities = np.log(opacities_clipped / (1.0 - opacities_clipped))

    n = means.shape[0]
    attrs = np.zeros((n, 3 + 3 + 3 + 1 + 3 + 4), dtype=np.float32)
    attrs[:, 0:3]   = means
    attrs[:, 3:6]   = 0.0
    attrs[:, 6:9]   = colors
    attrs[:, 9:10]  = logit_opacities[:, None]
    attrs[:, 10:13] = log_scales
    attrs[:, 13:17] = quats

    header = (
        'ply\n'
        'format binary_little_endian 1.0\n'
        f'element vertex {n}\n'
        'property float x\n'
        'property float y\n'
        'property float z\n'
        'property float nx\n'
        'property float ny\n'
        'property float nz\n'
        'property float f_dc_0\n'
        'property float f_dc_1\n'
        'property float f_dc_2\n'
        'property float opacity\n'
        'property float scale_0\n'
        'property float scale_1\n'
        'property float scale_2\n'
        'property float rot_0\n'
        'property float rot_1\n'
        'property float rot_2\n'
        'property float rot_3\n'
        'end_header\n'
    ).encode('ascii')

    with open(path, 'wb') as f:
        f.write(header)
        f.write(attrs.tobytes())

# --- Save the camera + intrinsics JSON (matches upstream save_camera_params) ─
def save_camera_params_json(c2w: np.ndarray, K: np.ndarray, out_dir: str):
    """Write camera_poses.json with shape metadata so it round-trips."""
    c2w = _to_numpy(c2w)
    K = _to_numpy(K)
    payload = {
        'c2w':  c2w.tolist(),
        'K':    K.tolist(),
        'count': int(c2w.shape[0]),
        'note': 'c2w is OpenCV camera-to-world (4x4 per frame); K is 3x3 intrinsics',
    }
    with open(os.path.join(out_dir, 'camera_poses.json'), 'w') as f:
        json.dump(payload, f, indent=2)

# --- Sky-segmentation ONNX loader ──────────────────────────────────────
# The vendored compute_sky_mask() downloads `skyseg.onnx` to the *current
# working directory* on first call.  We pre-download it into a stable
# location and chdir to that directory when calling compute_sky_mask
# (then chdir back) so we don't pollute /content with a stray file.
SKYSEG_DIR  = pathlib.Path('/content/hyworld_work/skyseg')
SKYSEG_DIR.mkdir(parents=True, exist_ok=True)
SKYSEG_PATH = SKYSEG_DIR / 'skyseg.onnx'
SKYSEG_URL  = 'https://huggingface.co/JianyuanWang/skyseg/resolve/main/skyseg.onnx'

def ensure_skyseg_onnx(verbose=True):
    """Pre-download skyseg.onnx into SKYSEG_DIR.  Idempotent."""
    if SKYSEG_PATH.exists() and SKYSEG_PATH.stat().st_size > 100_000:
        return str(SKYSEG_PATH)
    if verbose:
        print(f'  [skyseg] downloading {SKYSEG_URL} -> {SKYSEG_PATH}')
    r = requests.get(SKYSEG_URL, stream=True, timeout=60)
    r.raise_for_status()
    with open(SKYSEG_PATH, 'wb') as f:
        for chunk in r.iter_content(chunk_size=1 << 20):
            f.write(chunk)
    return str(SKYSEG_PATH)

def load_skyseg_onnx(verbose=True):
    """Pre-download the vendored skyseg.onnx and return its directory.

    Returns: str(path to SKYSEG_DIR) — pass this as the cwd when calling
    `compute_sky_mask()` (it looks for `skyseg.onnx` in the CWD).
    """
    ensure_skyseg_onnx(verbose=verbose)
    return str(SKYSEG_DIR)

# --- CUDA free + timing helpers ────────────────────────────────────────
def free_cuda(verbose=False):
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        if verbose:
            free, total = torch.cuda.mem_get_info()
            print(f'  [cuda] free={free/1024**3:.1f} GB / total={total/1024**3:.1f} GB')

def log_inference_time(t0, n_views):
    dt = time.perf_counter() - t0
    print(f'  Inference: {dt:.2f}s for {n_views} view(s)  ({dt/max(1,n_views):.3f}s / view)')

print('STEP 3 complete — helpers (extract, mask, save_gs_ply, sky onnx) ready.')


In [ ]:
#@title STEP 4 — Gradio 6.9.0 UI (gradio_rerun 3D viewer + per-view depth/normal tabs)
"""
• Two-column layout: left = input controls, right = 3 Rerun tabs +
  per-view depth + per-view normal galleries with prev/next navigation.
• Tabs: 3D Reconstruction (Rerun) | Gaussian Splats (Rerun) | Depth Maps |
  Normal Maps.
• Downloads: GLB mesh + Gaussian PLY + Rerun recording + zipped folder.
• Uses the orange-red theme (matches the official Space).
• Concurrency limit = 2 (matches the rest of the AEI suite).
• `clear_output(wait=True)` before launch.
• `demo.load` welcome message.

NOTE: In Gradio 6.x, `theme` / `css` are passed to `.launch()` rather than
`gr.Blocks()`.  We pin gradio==6.9.0 (the latest published 6.x on PyPI;
6.12.0 referenced by the Space doesn't exist).
"""
import os, sys, time, json, gc, uuid, shutil, traceback, zipfile
import pathlib
import numpy as np
import cv2
import torch
from PIL import Image
from glob import glob
from datetime import datetime
from typing import Optional

import gradio as gr
from gradio.themes import Soft
from gradio.themes.utils import colors, fonts, sizes
from gradio_rerun import Rerun
import rerun as rr
try:
    import rerun.blueprint as rrb
except ImportError:
    rrb = None

# --- Theme (orange-red, matches the official Space) ────────────────────
colors.orange_red = colors.Color(
    name='orange_red',
    c50='#FFF0E5', c100='#FFE0CC', c200='#FFC299', c300='#FFA366',
    c400='#FF8533', c500='#FF4500', c600='#E63E00', c700='#CC3700',
    c800='#B33000', c900='#992900', c950='#802200',
)
class OrangeRedTheme(Soft):
    def __init__(self, **kwargs):
        super().__init__(
            primary_hue=kwargs.get('primary_hue', colors.gray),
            secondary_hue=kwargs.get('secondary_hue', colors.orange_red),
            neutral_hue=kwargs.get('neutral_hue', colors.slate),
            text_size=kwargs.get('text_size', sizes.text_lg),
            font=kwargs.get('font', (fonts.GoogleFont('Outfit'), 'Arial', 'sans-serif')),
            font_mono=kwargs.get('font_mono', (fonts.GoogleFont('IBM Plex Mono'), 'ui-monospace', 'monospace')),
        )
        super().set(
            background_fill_primary='*primary_50',
            background_fill_primary_dark='*primary_900',
            body_background_fill='linear-gradient(135deg, *primary_200, *primary_100)',
            body_background_fill_dark='linear-gradient(135deg, *primary_900, *primary_800)',
            button_primary_text_color='white',
            button_primary_text_color_hover='white',
            button_primary_background_fill='linear-gradient(90deg, *secondary_500, *secondary_600)',
            button_primary_background_fill_hover='linear-gradient(90deg, *secondary_600, *secondary_700)',
            slider_color='*secondary_500',
            slider_color_dark='*secondary_600',
            block_title_text_weight='600',
            block_border_width='3px',
            block_shadow='*shadow_drop_lg',
            button_primary_shadow='*shadow_drop_lg',
            button_large_padding='11px',
            color_accent_soft='*primary_100',
            block_label_background_fill='*primary_200',
        )

orange_red_theme = OrangeRedTheme()

CSS = """
#col-container   { margin: 0 auto; max-width: 1500px; }
#main-title h1   { font-size: 2.4em !important; }
.viewer-card     { border-radius: 12px; overflow: hidden; }
"""

# --- Wrappers around the Space's Rerun recorders (same logic) ──────────
def _make_rec(app_id: str) -> 'rr.RecordingStream':
    run_id = str(uuid.uuid4())
    if hasattr(rr, 'new_recording'):
        return rr.new_recording(application_id=app_id, recording_id=run_id)
    return rr.RecordingStream(application_id=app_id, recording_id=run_id)

def build_reconstruction_rrd(output_subdir, glb_path, world_points, images_np,
                              camera_poses, camera_intrs, filter_mask, normals):
    rrd_path = str(output_subdir / 'reconstruction.rrd')
    rec = _make_rec('HY-World-2.0-Reconstruction')
    rec.log('world', rr.Clear(recursive=True), static=True)
    rec.log('world', rr.ViewCoordinates.RIGHT_HAND_Y_UP, static=True)
    try:
        rec.log('world/axes/x', rr.Arrows3D(vectors=[[0.5, 0, 0]], colors=[[220, 50, 50]]), static=True)
        rec.log('world/axes/y', rr.Arrows3D(vectors=[[0, 0.5, 0]], colors=[[50, 200, 50]]), static=True)
        rec.log('world/axes/z', rr.Arrows3D(vectors=[[0, 0, 0.5]], colors=[[50, 100, 220]]), static=True)
    except Exception:
        pass
    if glb_path and pathlib.Path(glb_path).exists():
        try:
            rec.log('world/scene_mesh', rr.Asset3D(path=str(glb_path)), static=True)
        except Exception as e:
            print(f'[Rerun] GLB log failed: {e}')
    S = images_np.shape[0] if images_np.ndim == 4 else 1
    for i in range(S):
        img_hwc = np.transpose(images_np[i], (1, 2, 0))
        img_u8 = (np.clip(img_hwc, 0, 1) * 255).astype(np.uint8)
        H, W = img_u8.shape[:2]
        mask = filter_mask[i] if (filter_mask and i < len(filter_mask)) else None
        if world_points.ndim in (4, 5):
            pts = world_points[i]
        else:
            pts = world_points
            mask = None
        if pts.ndim == 3:
            pts_f = pts.reshape(-1, 3)
            col_f = img_u8.reshape(-1, 3)
            if mask is not None:
                mf = mask.reshape(-1)
                pts_f = pts_f[mf]
                col_f = col_f[mf]
        else:
            pts_f = pts
            col_f = None
        if pts_f.shape[0] > 0:
            try:
                rec.log(
                    f'world/point_cloud/view_{i:03d}',
                    rr.Points3D(positions=pts_f.astype(np.float32), colors=col_f, radii=0.003),
                    static=True,
                )
            except Exception as e:
                print(f'[Rerun] Points3D view {i} failed: {e}')
        if camera_poses is not None and i < len(camera_poses):
            c2w = camera_poses[i]
            intr = camera_intrs[i] if camera_intrs is not None else None
            try:
                rec.log(f'world/cameras/cam_{i:03d}',
                        rr.Transform3D(translation=c2w[:3, 3].astype(np.float32),
                                       mat3x3=c2w[:3, :3].astype(np.float32)),
                        static=True)
                if intr is not None:
                    rec.log(
                        f'world/cameras/cam_{i:03d}/pinhole',
                        rr.Pinhole(
                            focal_length=[float(intr[0, 0]), float(intr[1, 1])],
                            principal_point=[float(intr[0, 2]), float(intr[1, 2])],
                            width=W, height=H,
                        ),
                        static=True,
                    )
                    rec.log(f'world/cameras/cam_{i:03d}/pinhole/image', rr.Image(img_u8), static=True)
            except Exception as e:
                print(f'[Rerun] Camera {i} failed: {e}')
        if normals is not None and i < len(normals):
            try:
                rec.log(f'world/normals/view_{i:03d}', rr.Image(render_normal_colormap(normals[i], mask)), static=True)
            except Exception as e:
                print(f'[Rerun] Normal {i} failed: {e}')
    if rrb is not None:
        try:
            bp = rrb.Blueprint(
                rrb.Horizontal(
                    rrb.Spatial3DView(origin='/world', name='3D Scene'),
                    rrb.Vertical(
                        rrb.Spatial2DView(origin=f'/world/cameras/cam_000/pinhole/image', name='Camera View'),
                        rrb.Spatial2DView(origin='/world/normals/view_000', name='Normal Map'),
                    ),
                    column_shares=[3, 1],
                ),
                collapse_panels=True,
            )
            rec.send_blueprint(bp)
        except Exception as e:
            print(f'[Rerun] Reconstruction blueprint failed (non-fatal): {e}')
    rec.save(rrd_path)
    return rrd_path

def build_gaussians_rrd(output_subdir, means, colors_np, opacities, scales):
    rrd_path = str(output_subdir / 'gaussians.rrd')
    rec = _make_rec('HY-World-2.0-Gaussians')
    means_c = means.copy()
    means_c[:, 1] = -means_c[:, 1]
    rec.log('world', rr.Clear(recursive=True), static=True)
    rec.log('world', rr.ViewCoordinates.RIGHT_HAND_Y_UP, static=True)
    # Rerun's Points3D expects sRGB colors, not raw SH coefficients.  Decode
    # the DC SH term to RGB using the standard SH_C0 constant so the
    # in-browser splat preview matches what SuperSplat/PlayCanvas will
    # show when opening the PLY file.
    if colors_np.dtype == np.uint8:
        colors_u8 = colors_np
    else:
        SH_C0 = 0.28209479177387814
        colors_rgb = 0.5 + SH_C0 * colors_np
        colors_u8 = (np.clip(colors_rgb, 0, 1) * 255).astype(np.uint8)
    alpha = (np.clip(opacities, 0, 1) * 255).astype(np.uint8).reshape(-1, 1)
    colors_rgba = np.concatenate([colors_u8, alpha], axis=1)
    radii = np.clip(np.linalg.norm(scales, axis=1) * 0.5, 0.001, 0.1).astype(np.float32)
    MAX_PTS = 2_000_000
    N = means_c.shape[0]
    if N > MAX_PTS:
        sel = np.random.default_rng(42).choice(N, size=MAX_PTS, replace=False)
        means_c = means_c[sel]; colors_rgba = colors_rgba[sel]; radii = radii[sel]
    try:
        rec.log('world/gaussian_splats',
                rr.Points3D(positions=means_c.astype(np.float32), colors=colors_rgba, radii=radii),
                static=True)
    except Exception as e:
        print(f'[Rerun] Gaussians Points3D failed: {e}')
    rec.save(rrd_path)
    return rrd_path

# --- Import the Space inference helpers (now patched) ──────────────────
from hyworldmirror.utils.inference_utils import (
    prepare_images_to_tensor,
    compute_adaptive_target_size,
    compute_sky_mask,
    compute_filter_mask,
    _voxel_prune_gaussians,
    depth_to_world_coords_points,
)
from hyworldmirror.utils.visual_util import convert_predictions_to_glb_scene

# The HF Space version of this function is decorated with @spaces.GPU
# for ZeroGPU duration capping.  We omit the decorator in Colab because
# the model runs directly on the GPU and a 120 s cap would just hurt UX.
def run_reconstruction_pipeline(target_dir, frame_selector,
                                 show_camera, filter_sky_bg, show_mesh, filter_ambiguous,
                                 max_target_size, confidence_percentile, max_gaussians,
                                 progress=gr.Progress(track_tqdm=True)):
    if not target_dir:
        raise gr.Error('Please upload files first before running reconstruction.')
    image_folder = pathlib.Path(target_dir) / 'images'
    img_paths = sorted(
        glob(str(image_folder / '*.png'))  +
        glob(str(image_folder / '*.jpg'))  +
        glob(str(image_folder / '*.jpeg')) +
        glob(str(image_folder / '*.webp'))
    )
    if not img_paths:
        raise gr.Error('No valid images found.  Please re-upload your files.')

    progress(0.10, desc='Loading model...')
    model, device, _ = load_worldmirror()
    effective = compute_adaptive_target_size(img_paths, max_target_size=int(max_target_size))

    progress(0.20, desc='Running inference...')
    with torch.no_grad():
        imgs = prepare_images_to_tensor(img_paths, target_size=effective, resize_strategy='crop').to(device)
        views = {'img': imgs}
        B, S, C, H, W = imgs.shape
        use_amp = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
        t0 = time.perf_counter()
        with torch.amp.autocast('cuda', enabled=use_amp, dtype=torch.bfloat16):
            predictions = model(views=views, cond_flags=[0, 0, 0], is_inference=True)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        infer_time = time.perf_counter() - t0
        log_inference_time(t0, S)

        progress(0.45, desc='Computing masks...')
        # Pre-download the upstream skyseg.onnx into a stable directory and
        # chdir into it for the compute_sky_mask() call (the vendored
        # function looks for `skyseg.onnx` in CWD).  We chdir back so
        # subsequent file writes land in the right place.
        try:
            ensure_skyseg_onnx(verbose=False)
        except Exception as e:
            print(f'  [sky] ONNX pre-download failed ({e}); will skip sky mask')
        try:
            old_cwd = os.getcwd()
            os.chdir(SKYSEG_DIR)
            try:
                sky_mask = compute_sky_mask(
                    img_paths, H, W, S, predictions=predictions,
                    source='auto',
                    model_threshold=0.45,
                    processed_aspect_ratio=W / H,
                )
            finally:
                os.chdir(old_cwd)
        except Exception as e:
            traceback.print_exc(limit=2)
            print(f'  [sky] mask computation failed: {e}; using a zero mask')
            sky_mask = np.ones((S, H, W), dtype=bool)

        filter_mask, gs_filter_mask = compute_filter_mask(
            predictions, imgs, img_paths, H, W, S,
            apply_confidence_mask=False,
            apply_edge_mask=True,
            apply_sky_mask=True,
            confidence_percentile=float(confidence_percentile),
            edge_normal_threshold=1.0,
            edge_depth_threshold=0.03,
            sky_mask=sky_mask,
            use_gs_depth=('gs_depth' in predictions),
        )

        output_id = uuid.uuid4().hex[:8]
        output_subdir = OUT_DIR / f'recon_{output_id}'
        output_subdir.mkdir(parents=True, exist_ok=True)
        safe_name = str(frame_selector).replace('.', '_').replace(':', '').replace(' ', '_')
        glb_path = output_subdir / f'scene_{safe_name}.glb'

        glb_preds = {}
        if 'world_points' not in predictions:
            glb_preds['world_points'] = depth_to_world_coords_points(
                predictions['depth'][0, ..., 0],
                predictions['camera_poses'][0],
                predictions['camera_intrs'][0],
            )[0].cpu().float().numpy()
        else:
            pts = predictions['world_points']
            if isinstance(pts, torch.Tensor):
                pts = pts.detach().cpu().numpy()
            if pts.ndim == 5:
                pts = pts[0]
            glb_preds['world_points'] = pts
        glb_preds['images'] = imgs[0].detach().cpu().numpy()
        c2w = predictions['camera_poses']
        if isinstance(c2w, torch.Tensor):
            c2w = c2w.detach().cpu().numpy()
        if c2w.ndim == 4:
            c2w = c2w[0]
        glb_preds['camera_poses'] = c2w
        glb_preds['final_mask'] = np.array(filter_mask)
        glb_preds['sky_mask']   = np.array(sky_mask)

        normals_np = None
        for key in ('normal', 'normals'):
            if key in predictions and predictions[key] is not None:
                n = predictions[key]
                if isinstance(n, torch.Tensor):
                    n = n.detach().cpu().numpy()
                if n.ndim == 5:
                    n = n[0]
                glb_preds['normal'] = n
                normals_np = n
                break

        progress(0.60, desc='Building GLB mesh...')
        glb_scene = convert_predictions_to_glb_scene(
            glb_preds,
            filter_by_frames=frame_selector,
            show_camera=show_camera,
            mask_sky_bg=filter_sky_bg,
            as_mesh=show_mesh,
            mask_ambiguous=filter_ambiguous,
        )
        glb_scene.export(file_obj=str(glb_path))

        progress(0.70, desc='Building Rerun reconstruction recording...')
        recon_rrd = build_reconstruction_rrd(
            output_subdir=output_subdir,
            glb_path=glb_path,
            world_points=glb_preds['world_points'],
            images_np=glb_preds['images'],
            camera_poses=glb_preds['camera_poses'],
            camera_intrs=predictions['camera_intrs'][0].cpu().float().numpy(),
            filter_mask=list(filter_mask),
            normals=normals_np,
        )

        progress(0.78, desc='Rendering depth and normal maps...')
        depth_imgs, normal_imgs = [], []
        for i in range(S):
            depth = predictions['depth'][0, i].cpu().float().numpy().squeeze()
            mask = filter_mask[i] if i < len(filter_mask) else None
            depth_p = output_subdir / f'depth_{i:03d}.png'
            Image.fromarray(render_depth_colormap(depth, mask)).save(depth_p)
            depth_imgs.append(str(depth_p))
            np.save(output_subdir / f'depth_{i:03d}.npy', depth.astype(np.float32))
            normal = np.zeros((H, W, 3), dtype=np.float32)
            for k in ('normals', 'normal'):
                if k in predictions and predictions[k] is not None:
                    normal = predictions[k][0, i].cpu().float().numpy()
                    break
            normal_p = output_subdir / f'normal_{i:03d}.png'
            Image.fromarray(render_normal_colormap(normal, mask)).save(normal_p)
            normal_imgs.append(str(normal_p))

        save_camera_params_json(
            predictions['camera_poses'][0].cpu().float().numpy(),
            predictions['camera_intrs'][0].cpu().float().numpy(),
            str(output_subdir),
        )

        progress(0.88, desc='Processing Gaussian splats...')
        gs_rrd = None
        gs_ply = None
        if 'splats' in predictions:
            sp = predictions['splats']
            means     = sp['means'][0].reshape(-1, 3).cpu()
            scales    = sp['scales'][0].reshape(-1, 3).cpu()
            quats     = sp['quats'][0].reshape(-1, 4).cpu()
            colors_t  = (sp.get('sh', sp.get('colors'))[0]).reshape(-1, 3).cpu()
            opacities = sp['opacities'][0].reshape(-1).cpu()
            weights   = sp['weights'][0].reshape(-1).cpu() if 'weights' in sp else torch.ones_like(opacities)
            if gs_filter_mask is not None:
                keep = torch.from_numpy(np.asarray(gs_filter_mask).reshape(-1)).bool()
                means, scales, quats = means[keep], scales[keep], quats[keep]
                colors_t, opacities, weights = colors_t[keep], opacities[keep], weights[keep]
            means, scales, quats, colors_t, opacities = _voxel_prune_gaussians(
                means, scales, quats, colors_t, opacities, weights,
            )
            if means.shape[0] > int(max_gaussians):
                idx = torch.from_numpy(
                    np.random.default_rng(42).choice(means.shape[0], int(max_gaussians), replace=False)
                ).long()
                means, scales, quats, colors_t, opacities = (
                    means[idx], scales[idx], quats[idx], colors_t[idx], opacities[idx]
                )
            ply_out = str(output_subdir / 'gaussians.ply')
            save_gs_ply_aes(ply_out, means, scales, quats, colors_t, opacities)
            gs_ply = ply_out
            progress(0.94, desc='Building Rerun recording for Gaussian splats...')
            gs_rrd = build_gaussians_rrd(
                output_subdir=output_subdir,
                means=means.float().numpy(),
                colors_np=colors_t.float().numpy(),
                opacities=opacities.float().numpy(),
                scales=scales.float().numpy(),
            )

        del predictions, glb_preds, imgs
        free_cuda()

        # Bundle the entire output folder into a single zip so the
        # "Download folder" button can serve a valid path on the first click.
        zip_path = output_subdir / 'outputs.zip'
        if zip_path.exists():
            zip_path.unlink()
        with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=4) as zf:
            for p in output_subdir.rglob('*'):
                if p.is_file() and p != zip_path:
                    zf.write(p, arcname=p.relative_to(output_subdir))

        status = (
            f'Reconstruction complete — {S} view(s), '
            f'inference time: {infer_time:.2f}s, '
            f'output: {output_subdir.name}'
        )
        return (
            recon_rrd, depth_imgs, depth_imgs[0]  if depth_imgs  else None,
            normal_imgs, normal_imgs[0] if normal_imgs else None,
            status, gs_rrd, gs_ply, str(output_subdir), str(glb_path), str(zip_path),
        )

# --- Gradio UI block construction ──────────────────────────────────────
with gr.Blocks(delete_cache=(600, 600)) as demo:
    gr.Markdown(
        '# **HY-World 2.0 — WorldMirror 2.0 World Reconstruction**',
        elem_id='main-title',
    )
    gr.Markdown(
        'Upload images or a short video to reconstruct a full 3D scene: '
        'point clouds, camera poses, depth maps, surface normals and Gaussian splats — '
        'powered by [HY-World 2.0 / WorldMirror](https://huggingface.co/tencent/HY-World-2.0). '
        'Weights are under the Tencent HY-WORLD 2.0 Community License (territory excludes EU/UK/KR).'
    )

    target_dir_state  = gr.State(None)
    depth_list_state  = gr.State([])
    normal_list_state = gr.State([])
    depth_idx_state   = gr.State(0)
    normal_idx_state  = gr.State(0)
    out_subdir_state  = gr.State('')
    glb_path_state    = gr.State('')
    gs_ply_state      = gr.State('')
    rrd_state         = gr.State('')

    with gr.Row(elem_id='col-container'):
        with gr.Column(scale=1, min_width=380):
            file_upload = gr.File(
                label='Upload images or a short video',
                file_count='multiple',
                file_types=['image', 'video', '.heic', '.heif'],
                info='Drag-and-drop one or more images. For a video, frames are extracted at the interval below.',
            )
            input_gallery = gr.Gallery(
                label='Input frames preview',
                columns=2, rows=2, height=220, object_fit='cover', preview=False,
            )
            time_interval = gr.Slider(
                0.1, 10.0, value=1.0, step=0.1,
                label='Video interval (seconds)',
                info='Only used when the upload contains a video. 1.0 = one frame per second.',
            )
            frame_selector = gr.Dropdown(
                choices=['All'], value='All', label='Frame selector',
                info='"All" uses every extracted frame. Switch to a single index to rebuild a GLB of just that view.',
            )
            with gr.Accordion('Reconstruction options', open=False):
                show_camera      = gr.Checkbox(value=True,  label='Show camera frustums',     info='Render a wireframe camera per input frame inside the Rerun viewer.')
                show_mesh        = gr.Checkbox(value=True,  label='Show as triangulated mesh', info='If off, exports a colored point cloud instead. Mesh is slower to build.')
                filter_ambiguous = gr.Checkbox(value=True,  label='Filter low-confidence / edges', info='Drop points that lie on depth/normal discontinuities.')
                filter_sky_bg    = gr.Checkbox(value=False, label='Filter sky background',    info='Mask out the sky on every frame before triangulation. Requires the skyseg ONNX.')
                max_target_size  = gr.Slider(256, 1024, value=504, step=14, label='Max image side (px)', info='Multiples of 14 (patch size). Higher = sharper but more VRAM. Drop to 322 on T4.')
                confidence_pct   = gr.Slider(1.0, 50.0, value=10.0, step=1.0, label='Confidence percentile cutoff', info='Drop the bottom N% of pixels by per-view depth confidence.')
                max_gaussians    = gr.Slider(100_000, 5_000_000, value=2_000_000, step=100_000, label='Max gaussians (cap)', info='Subsample the splat cloud above this many points. Default 2M fits in any viewer.')
            btn_reconstruct = gr.Button('Reconstruct 3D scene', variant='primary')
            status_box = gr.Textbox(label='Status', interactive=False, lines=2, visible=False, placeholder='Awaiting reconstruction...')
            with gr.Accordion('Downloads', open=False):
                dl_glb    = gr.DownloadButton(label='Download GLB mesh',         variant='secondary', visible=False)
                dl_gs_ply = gr.DownloadButton(label='Download Gaussian PLY',     variant='secondary', visible=False)
                dl_rrd    = gr.DownloadButton(label='Download Rerun recording',  variant='secondary', visible=False)
                dl_outdir = gr.DownloadButton(label='Download output folder (zip)', variant='secondary', visible=False)

        with gr.Column(scale=2, elem_id='viewers-row'):
            with gr.Tabs():
                with gr.Tab('3D reconstruction'):
                    gr.Markdown('### 3D scene — Rerun viewer (point cloud + cameras + normals)')
                    rerun_recon = Rerun(label='3D reconstruction — Rerun viewer', height=620)
                with gr.Tab('Gaussian splats'):
                    gr.Markdown('### Gaussian-splat point cloud — Rerun viewer (also downloadable as `.ply`)')
                    rerun_gs = Rerun(label='Gaussian splats — Rerun viewer', height=620)
                    gr.Markdown('The PLY file (downloadable on the left) opens directly in [SuperSplat](https://supersplat.xyz) or PlayCanvas for full splat rendering.')
                with gr.Tab('Depth maps'):
                    gr.Markdown('### Per-view depth maps (turbo colormap + raw `.npy` saved to the output folder)')
                    depth_image = gr.Image(label='Depth map', type='filepath', height=520, interactive=False)
                    with gr.Row():
                        btn_depth_prev = gr.Button('Previous', variant='secondary', scale=1)
                        depth_counter  = gr.Textbox(value='–', label='Frame', interactive=False, scale=1)
                        btn_depth_next = gr.Button('Next',     variant='secondary', scale=1)
                with gr.Tab('Normal maps'):
                    gr.Markdown('### Per-view surface normal maps')
                    normal_image = gr.Image(label='Normal map', type='filepath', height=520, interactive=False)
                    with gr.Row():
                        btn_normal_prev = gr.Button('Previous', variant='secondary', scale=1)
                        normal_counter  = gr.Textbox(value='–', label='Frame', interactive=False, scale=1)
                        btn_normal_next = gr.Button('Next',     variant='secondary', scale=1)

    # --- Event wiring ---
    def on_files_uploaded(files, time_interval):
        if not files:
            return [], None, gr.update(choices=['All'], value='All')
        target_dir, img_paths = process_uploaded_files(files, time_interval)
        gallery_items = [(p, pathlib.Path(p).stem) for p in img_paths]
        choices = ['All'] + [str(i) for i in range(len(img_paths))]
        return gallery_items, target_dir, gr.update(choices=choices, value='All')

    file_upload.upload(
        on_files_uploaded,
        inputs=[file_upload, time_interval],
        outputs=[input_gallery, target_dir_state, frame_selector],
    )
    time_interval.change(
        on_files_uploaded,
        inputs=[file_upload, time_interval],
        outputs=[input_gallery, target_dir_state, frame_selector],
    )

    def reconstruct(target_dir, frame_selector, show_camera, filter_sky_bg, show_mesh,
                     filter_ambiguous, max_target_size, confidence_pct, max_gaussians,
                     progress=gr.Progress(track_tqdm=True)):
        try:
            (recon_rrd, depth_list, depth_first, normal_list, normal_first, status,
             gs_rrd, gs_ply, out_subdir, glb_path, zip_path) = run_reconstruction_pipeline(
                target_dir, frame_selector, show_camera, filter_sky_bg, show_mesh, filter_ambiguous,
                max_target_size, confidence_pct, max_gaussians, progress=progress,
            )
            return (
                recon_rrd,
                depth_list, depth_first, depth_counter_text(depth_list, 0),
                normal_list, normal_first, normal_counter_text(normal_list, 0),
                status, gr.update(visible=True, value=status),
                gs_rrd,
                gr.update(visible=True, value=glb_path) if glb_path else gr.update(visible=False),
                gr.update(visible=True, value=gs_ply)  if gs_ply  else gr.update(visible=False),
                gr.update(visible=True, value=recon_rrd) if recon_rrd else gr.update(visible=False),
                gr.update(visible=True, value=zip_path) if zip_path else gr.update(visible=False),
                out_subdir, glb_path, gs_ply or '', recon_rrd or '',
                0, 0,  # reset depth/normal idx
            )
        except Exception as e:
            traceback.print_exc(limit=4)
            raise gr.Error(f'Reconstruction failed: {e}')

    def depth_counter_text(items, idx):
        if not items:
            return '–'
        return f'{int(idx)} / {len(items)-1}'

    btn_reconstruct.click(
        reconstruct,
        inputs=[target_dir_state, frame_selector,
                show_camera, filter_sky_bg, show_mesh, filter_ambiguous,
                max_target_size, confidence_pct, max_gaussians],
        outputs=[
            rerun_recon,
            depth_list_state, depth_image, depth_counter,
            normal_list_state, normal_image, normal_counter,
            status, status_box,
            rerun_gs,
            dl_glb, dl_gs_ply, dl_rrd, dl_outdir,
            out_subdir_state, glb_path_state, gs_ply_state, rrd_state,
            depth_idx_state, normal_idx_state,
        ],
    )

    # Prev/Next navigation for depth and normal tabs
    def step_image(items, idx, delta):
        if not items:
            return None, '–', 0
        new_idx = max(0, min(len(items) - 1, int(idx) + int(delta)))
        return items[new_idx], f'{new_idx} / {len(items)-1}', new_idx

    btn_depth_prev.click(lambda d, i: step_image(d, i, -1),
                          inputs=[depth_list_state, depth_idx_state],
                          outputs=[depth_image, depth_counter, depth_idx_state])
    btn_depth_next.click(lambda d, i: step_image(d, i, +1),
                          inputs=[depth_list_state, depth_idx_state],
                          outputs=[depth_image, depth_counter, depth_idx_state])
    btn_normal_prev.click(lambda d, i: step_image(d, i, -1),
                           inputs=[normal_list_state, normal_idx_state],
                           outputs=[normal_image, normal_counter, normal_idx_state])
    btn_normal_next.click(lambda d, i: step_image(d, i, +1),
                           inputs=[normal_list_state, normal_idx_state],
                           outputs=[normal_image, normal_counter, normal_idx_state])

    # The "Download output folder" button is wired to the cached zip
    # created at the end of run_reconstruction_pipeline.  No click handler
    # needed — the file is already on disk and the value is set by the
    # `reconstruct()` function above.

    def _welcome():
        return (
            'Upload one or more images (or a short video) on the left, then click '
            '"Reconstruct 3D scene". First run takes ~8 min for the 5 GB weight download; '
            'subsequent runs are ~30 s.'
        )
    demo.load(_welcome, inputs=None, outputs=[status])

# --- Queue + launch (theme + css are passed to launch() in Gradio 6.x) ─
demo.queue(default_concurrency_limit=2, max_size=8)
try:
    from IPython.display import clear_output
    clear_output()
    clear_output(wait=True)
except Exception:
    pass
demo.launch(
    share=False,
    server_name='0.0.0.0',
    server_port=7860,
    show_error=True,
    height=1100,
    theme=orange_red_theme,
    css=CSS,
)


In [ ]:
#@title STEP 5 — Keep the Colab runtime alive + session summary
"""
Standard AEI-suite keep-alive cell.  Hits a local URL once a minute via
`IPython.display.Javascript` to prevent the 90-minute idle disconnect.
Also prints a session summary so users can see where everything went.
"""
import os, sys, time, json, pathlib, datetime, traceback
import IPython
from IPython.display import display, Javascript

print('='*72)
print('Keep-alive timer started.  Run the next cell to keep the tab active.')
print('='*72)

# --- Session summary (read the cache + outputs) ────────────────────────
try:
    summary = {
        'cache_root'    : str(CACHE_ROOT),
        'work_root'     : str(WORK_ROOT),
        'repo_dir'      : str(REPO_DIR),
        'model_dir'     : str(MODEL_DIR),
        'out_dir'       : str(OUT_DIR),
        'license'       : LICENSE_NOTICE,
        'torch'         : torch.__version__,
        'cuda'          : torch.version.cuda,
        'gradio'        : gr.__version__,
        'gsplat'        : gsplat.__version__ if 'gsplat' in dir() else '?',
        'rerun_sdk'     : rr.__version__ if 'rr' in dir() else '?',
        'gpu'           : None,
    }
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        summary['gpu'] = f'{p.name}  ({p.total_memory / (1024**3):.1f} GB)'
    recon_count = len(list(OUT_DIR.glob('recon_*'))) if OUT_DIR.exists() else 0
    summary['recon_count'] = recon_count
    print('\n  Session summary')
    print('  ' + '-'*68)
    for k, v in summary.items():
        if isinstance(v, str) and len(v) > 80:
            v = v[:77] + '...'
        print(f'  {k:18s}: {v}')
except Exception as e:
    print(f'  WARN: summary print failed: {e}')

# --- Keep-alive timer ──────────────────────────────────────────────────
display(Javascript('''
function ClickConnect() {
  console.log("Keeping Colab alive — ", new Date().toLocaleTimeString());
  document.querySelector("colab-connect-button")?.click();
}
setInterval(ClickConnect, 60000);
'''))
print('\n  Keep-alive timer registered (60 s interval).  Safe to run the next cell.')


In [ ]:
#@title STEP 6 — Quick test (Colab-side file picker for a single image)
"""
Stand-alone test path.  Pick a single image, the model will reconstruct
it as a 1-view scene (depth + normals + a tiny splat cloud).  Useful for
verifying the install without opening the Gradio UI.

Also exposes a `FileLink` to the generated `outputs.zip` so the user can
download it from the Colab file browser without leaving the notebook.
"""
import os, sys, time, json, pathlib, traceback, shutil
import numpy as np
import torch
from PIL import Image
from IPython.display import display, FileLink

# This cell is meant to be run AFTER STEPS 1-3 (which set up the cache,
# imports, and lazy loader).  We re-import just the bits we need in case
# the user jumped straight here from STEP 1.
from hyworldmirror.utils.inference_utils import (
    prepare_images_to_tensor,
    compute_adaptive_target_size,
    compute_sky_mask,
    compute_filter_mask,
    _voxel_prune_gaussians,
    depth_to_world_coords_points,
)
from hyworldmirror.utils.visual_util import convert_predictions_to_glb_scene

print('='*72)
print('HY-World 2.0 — single-image quick test')
print('='*72)

# --- Use the Colab file-upload widget so we can grab a single file ────
try:
    from google.colab import files
    HAVE_COLAB_FILES = True
except ImportError:
    HAVE_COLAB_FILES = False
    print('  google.colab.files not available; set IMAGE_PATH manually below.')

if HAVE_COLAB_FILES:
    print('\n  Pick ONE image to test (PNG / JPG / WEBP / HEIC).')
    uploaded = files.upload()
    if not uploaded:
        raise SystemExit('No file uploaded — aborting quick test.')
    IMAGE_PATH = pathlib.Path(next(iter(uploaded.keys())))
else:
    IMAGE_PATH = pathlib.Path('/content/sample.jpg')
    if not IMAGE_PATH.exists():
        raise SystemExit(f'No image at {IMAGE_PATH} — please upload one manually.')

# --- Make a 1-image input folder ───────────────────────────────────────
target_dir = pathlib.Path('/content/hyworld_quicktest')
images_dir = target_dir / 'images'
if images_dir.exists():
    shutil.rmtree(images_dir)
images_dir.mkdir(parents=True, exist_ok=True)
dst = images_dir / IMAGE_PATH.name
if dst.resolve() != IMAGE_PATH.resolve():
    shutil.copy2(str(IMAGE_PATH), str(dst))
img_paths = [str(dst)]
print(f'  Image: {img_paths[0]}')

# --- Lazy-load the model and run inference ────────────────────────────
model, device, _ = load_worldmirror()
effective = compute_adaptive_target_size(img_paths, max_target_size=504)
print(f'  Effective resize: {effective} px (must be a multiple of 14)')

with torch.no_grad():
    imgs = prepare_images_to_tensor(img_paths, target_size=effective, resize_strategy='crop').to(device)
    views = {'img': imgs}
    B, S, C, H, W = imgs.shape
    use_amp = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    t0 = time.perf_counter()
    with torch.amp.autocast('cuda', enabled=use_amp, dtype=torch.bfloat16):
        predictions = model(views=views, cond_flags=[0, 0, 0], is_inference=True)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    infer_time = time.perf_counter() - t0
    log_inference_time(t0, S)

    # Quick mask (no ONNX, no edges) so the output is a clean 1-view scene
    sky_mask = [np.ones((H, W), dtype=bool) for _ in range(S)]
    filter_mask, _ = compute_filter_mask(
        predictions, imgs, img_paths, H, W, S,
        apply_confidence_mask=False,
        apply_edge_mask=True,
        apply_sky_mask=False,
        confidence_percentile=10.0,
        edge_normal_threshold=1.0,
        edge_depth_threshold=0.03,
        sky_mask=sky_mask,
        use_gs_depth=('gs_depth' in predictions),
    )

    out_id = f'quicktest_{int(time.time())}'
    out_subdir = OUT_DIR / out_id
    out_subdir.mkdir(parents=True, exist_ok=True)

    glb_preds = {}
    if 'world_points' not in predictions:
        glb_preds['world_points'] = depth_to_world_coords_points(
            predictions['depth'][0, ..., 0],
            predictions['camera_poses'][0],
            predictions['camera_intrs'][0],
        )[0].cpu().float().numpy()
    else:
        pts = predictions['world_points']
        if isinstance(pts, torch.Tensor):
            pts = pts.detach().cpu().numpy()
        if pts.ndim == 5:
            pts = pts[0]
        glb_preds['world_points'] = pts
    glb_preds['images']        = imgs[0].detach().cpu().numpy()
    c2w = predictions['camera_poses']
    if isinstance(c2w, torch.Tensor):
        c2w = c2w.detach().cpu().numpy()
    if c2w.ndim == 4:
        c2w = c2w[0]
    glb_preds['camera_poses']  = c2w
    glb_preds['final_mask']    = np.array(filter_mask)
    glb_preds['sky_mask']      = np.array(sky_mask)

    normals_np = None
    for key in ('normal', 'normals'):
        if key in predictions and predictions[key] is not None:
            n = predictions[key]
            if isinstance(n, torch.Tensor):
                n = n.detach().cpu().numpy()
            if n.ndim == 5:
                n = n[0]
            glb_preds['normal'] = n
            normals_np = n
            break

    glb_path = out_subdir / 'scene.glb'
    glb_scene = convert_predictions_to_glb_scene(
        glb_preds, filter_by_frames='All', show_camera=True, mask_sky_bg=False,
        as_mesh=True, mask_ambiguous=True,
    )
    glb_scene.export(file_obj=str(glb_path))

    # Depth + normal PNGs
    depth = predictions['depth'][0, 0].cpu().float().numpy().squeeze()
    Image.fromarray(render_depth_colormap(depth, filter_mask[0])).save(out_subdir / 'depth.png')
    np.save(out_subdir / 'depth.npy', depth.astype(np.float32))
    if normals_np is not None:
        n0 = normals_np[0]
        Image.fromarray(render_normal_colormap(n0, filter_mask[0])).save(out_subdir / 'normal.png')

    # Camera params
    save_camera_params_json(
        predictions['camera_poses'][0].cpu().float().numpy(),
        predictions['camera_intrs'][0].cpu().float().numpy(),
        str(out_subdir),
    )

    # 3DGS PLY
    if 'splats' in predictions:
        sp = predictions['splats']
        means     = sp['means'][0].reshape(-1, 3).cpu()
        scales    = sp['scales'][0].reshape(-1, 3).cpu()
        quats     = sp['quats'][0].reshape(-1, 4).cpu()
        colors_t  = (sp.get('sh', sp.get('colors'))[0]).reshape(-1, 3).cpu()
        opacities = sp['opacities'][0].reshape(-1).cpu()
        save_gs_ply_aes(str(out_subdir / 'gaussians.ply'), means, scales, quats, colors_t, opacities)

    del predictions, glb_preds, imgs
    free_cuda()

# --- Summary + download ────────────────────────────────────────────────
print()
print('='*72)
print(f'Single-image reconstruction complete in {infer_time:.2f}s')
print('='*72)
print(f'  Output folder : {out_subdir}')
for p in sorted(out_subdir.iterdir()):
    sz = p.stat().st_size
    if sz > 1024 * 1024:
        sz_s = f'{sz/1024/1024:.1f} MB'
    elif sz > 1024:
        sz_s = f'{sz/1024:.1f} KB'
    else:
        sz_s = f'{sz} B'
    print(f'    {p.name:>30s}  {sz_s:>10s}')

# Bundle the output into a zip for a one-click download
import zipfile
zip_path = out_subdir / 'quicktest.zip'
if zip_path.exists():
    zip_path.unlink()
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=4) as zf:
    for p in out_subdir.rglob('*'):
        if p.is_file() and p != zip_path:
            zf.write(p, arcname=p.relative_to(out_subdir))
print(f'\n  Zipped:        {zip_path}  ({zip_path.stat().st_size/1024/1024:.1f} MB)')

# Expose a Colab file-link for one-click download
display(FileLink(str(zip_path), result_html_prefix='Download quick-test results: '))
print('STEP 6 complete.  Open the Gradio UI (STEP 4) for the full multi-image experience.')


In [ ]:
#@title STEP 7 — Batch process a folder of image sets
"""
Drive-mode batch processor.  Point it at a folder that contains one
sub-folder per scene (each sub-folder = one reconstruction).  Useful
for sweeping through a dataset overnight on a paid Colab session.

Each sub-folder is processed independently:
  * Inference (4-15 s on A100 for 5-15 images)
  * 3DGS PLY + GLB mesh + camera poses + per-view depth + per-view normals
  * Rerun recording
  * Saved under  `OUT_DIR / recon_<scene>_<timestamp>/`

A progress log is written to `OUT_DIR / batch_log.jsonl` so you can
resume after a Colab disconnect — re-running this cell with the same
`BATCH_INPUT_FOLDER` skips scenes already logged.
"""
import os, sys, time, json, pathlib, shutil, traceback, uuid
import numpy as np
import torch
from PIL import Image
from glob import glob
from datetime import datetime

# Re-import the inference helpers in case the user ran only STEP 1 + 7
from hyworldmirror.utils.inference_utils import (
    prepare_images_to_tensor,
    compute_adaptive_target_size,
    compute_sky_mask,
    compute_filter_mask,
    _voxel_prune_gaussians,
    depth_to_world_coords_points,
)
from hyworldmirror.utils.visual_util import convert_predictions_to_glb_scene

print('='*72)
print('HY-World 2.0 — batch reconstruction')
print('='*72)

# --- Inputs ────────────────────────────────────────────────────────────
BATCH_INPUT_FOLDER = '/content/drive/MyDrive/AEI_3D_Cache/HY-World-2.0/batch_in'  #@param {type:'string'}
# Each sub-folder of BATCH_INPUT_FOLDER is one scene (must contain >=2
# images).  Nested sub-folders are recursed one level.
MAX_TARGET_SIZE_BATCH  = 504       #@param {type:'integer'}
SHOW_MESH              = True      #@param {type:'boolean'}
MAX_GAUSSIANS_PER_RUN  = 1_000_000 #@param {type:'integer'}
SKIP_COMPLETED         = True      #@param {type:'boolean'}

BATCH_INPUT_FOLDER = pathlib.Path(BATCH_INPUT_FOLDER).expanduser()
if not BATCH_INPUT_FOLDER.exists():
    print(f'  BATCH_INPUT_FOLDER does not exist: {BATCH_INPUT_FOLDER}')
    print(f'  Creating an empty one — drop your scene sub-folders in and re-run.')
    BATCH_INPUT_FOLDER.mkdir(parents=True, exist_ok=True)

batch_log_path = OUT_DIR / 'batch_log.jsonl'
already_done = set()
if SKIP_COMPLETED and batch_log_path.exists():
    try:
        with open(batch_log_path) as f:
            for ln in f:
                try:
                    rec = json.loads(ln)
                    already_done.add(rec.get('scene', ''))
                except Exception:
                    pass
    except Exception:
        pass
    print(f'  Resume mode: skipping {len(already_done)} already-processed scene(s)')

scenes = sorted([p for p in BATCH_INPUT_FOLDER.iterdir() if p.is_dir()])
if not scenes:
    print(f'  No scene sub-folders found under {BATCH_INPUT_FOLDER}')
    print('  Drop a folder structure like:')
    print('    batch_in/')
    print('      scene_001/  01.jpg 02.jpg 03.jpg ...')
    print('      scene_002/  ...')
    raise SystemExit(0)
print(f'  Found {len(scenes)} scene(s) in {BATCH_INPUT_FOLDER}')

# --- Load model once for the whole batch ──────────────────────────────
model, device, _ = load_worldmirror()

# --- Helpers ──────────────────────────────────────────────────────────
def _process_one_scene(scene_dir: pathlib.Path, scene_id: str):
    img_paths = sorted(
        glob(str(scene_dir / '*.png'))  +
        glob(str(scene_dir / '*.jpg'))  +
        glob(str(scene_dir / '*.jpeg')) +
        glob(str(scene_dir / '*.webp'))
    )
    if len(img_paths) < 2:
        return None, f'only {len(img_paths)} image(s) — need at least 2'
    effective = compute_adaptive_target_size(img_paths, max_target_size=int(MAX_TARGET_SIZE_BATCH))

    out_subdir = OUT_DIR / f'recon_{scene_id}_{int(time.time())}'
    out_subdir.mkdir(parents=True, exist_ok=True)

    with torch.no_grad():
        imgs = prepare_images_to_tensor(img_paths, target_size=effective, resize_strategy='crop').to(device)
        views = {'img': imgs}
        B, S, C, H, W = imgs.shape
        use_amp = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
        t0 = time.perf_counter()
        with torch.amp.autocast('cuda', enabled=use_amp, dtype=torch.bfloat16):
            predictions = model(views=views, cond_flags=[0, 0, 0], is_inference=True)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        infer_time = time.perf_counter() - t0

        sky_mask = np.ones((S, H, W), dtype=bool)
        try:
            ensure_skyseg_onnx(verbose=False)
            old_cwd = os.getcwd()
            os.chdir(SKYSEG_DIR)
            try:
                sky_mask = compute_sky_mask(
                    img_paths, H, W, S, predictions=predictions,
                    source='auto', model_threshold=0.45,
                    processed_aspect_ratio=W / H,
                )
            finally:
                os.chdir(old_cwd)
        except Exception as e:
            print(f'  [sky] mask failed for {scene_id}: {e}')
        filter_mask, gs_filter_mask = compute_filter_mask(
            predictions, imgs, img_paths, H, W, S,
            apply_confidence_mask=False,
            apply_edge_mask=True,
            apply_sky_mask=True,
            confidence_percentile=10.0,
            edge_normal_threshold=1.0,
            edge_depth_threshold=0.03,
            sky_mask=sky_mask,
            use_gs_depth=('gs_depth' in predictions),
        )

        glb_preds = {}
        if 'world_points' not in predictions:
            glb_preds['world_points'] = depth_to_world_coords_points(
                predictions['depth'][0, ..., 0],
                predictions['camera_poses'][0],
                predictions['camera_intrs'][0],
            )[0].cpu().float().numpy()
        else:
            pts = predictions['world_points']
            if isinstance(pts, torch.Tensor):
                pts = pts.detach().cpu().numpy()
            if pts.ndim == 5:
                pts = pts[0]
            glb_preds['world_points'] = pts
        glb_preds['images']       = imgs[0].detach().cpu().numpy()
        c2w = predictions['camera_poses']
        if isinstance(c2w, torch.Tensor):
            c2w = c2w.detach().cpu().numpy()
        if c2w.ndim == 4:
            c2w = c2w[0]
        glb_preds['camera_poses'] = c2w
        glb_preds['final_mask']   = np.array(filter_mask)
        glb_preds['sky_mask']     = np.array(sky_mask)
        normals_np = None
        for key in ('normal', 'normals'):
            if key in predictions and predictions[key] is not None:
                n = predictions[key]
                if isinstance(n, torch.Tensor):
                    n = n.detach().cpu().numpy()
                if n.ndim == 5:
                    n = n[0]
                glb_preds['normal'] = n
                normals_np = n
                break

        glb_path = out_subdir / f'scene_{scene_id}.glb'
        glb_scene = convert_predictions_to_glb_scene(
            glb_preds, filter_by_frames='All', show_camera=False,
            mask_sky_bg=True, as_mesh=bool(SHOW_MESH), mask_ambiguous=True,
        )
        glb_scene.export(file_obj=str(glb_path))

        recon_rrd = build_reconstruction_rrd(
            output_subdir=out_subdir,
            glb_path=glb_path,
            world_points=glb_preds['world_points'],
            images_np=glb_preds['images'],
            camera_poses=glb_preds['camera_poses'],
            camera_intrs=predictions['camera_intrs'][0].cpu().float().numpy(),
            filter_mask=list(filter_mask),
            normals=normals_np,
        )

        for i in range(S):
            depth = predictions['depth'][0, i].cpu().float().numpy().squeeze()
            mask = filter_mask[i] if i < len(filter_mask) else None
            Image.fromarray(render_depth_colormap(depth, mask)).save(out_subdir / f'depth_{i:03d}.png')
            np.save(out_subdir / f'depth_{i:03d}.npy', depth.astype(np.float32))
            if normals_np is not None:
                Image.fromarray(render_normal_colormap(normals_np[i], mask)).save(out_subdir / f'normal_{i:03d}.png')
        save_camera_params_json(
            predictions['camera_poses'][0].cpu().float().numpy(),
            predictions['camera_intrs'][0].cpu().float().numpy(),
            str(out_subdir),
        )

        if 'splats' in predictions:
            sp = predictions['splats']
            means     = sp['means'][0].reshape(-1, 3).cpu()
            scales    = sp['scales'][0].reshape(-1, 3).cpu()
            quats     = sp['quats'][0].reshape(-1, 4).cpu()
            colors_t  = (sp.get('sh', sp.get('colors'))[0]).reshape(-1, 3).cpu()
            opacities = sp['opacities'][0].reshape(-1).cpu()
            weights   = sp['weights'][0].reshape(-1).cpu() if 'weights' in sp else torch.ones_like(opacities)
            if gs_filter_mask is not None:
                keep = torch.from_numpy(np.asarray(gs_filter_mask).reshape(-1)).bool()
                means, scales, quats = means[keep], scales[keep], quats[keep]
                colors_t, opacities, weights = colors_t[keep], opacities[keep], weights[keep]
            means, scales, quats, colors_t, opacities = _voxel_prune_gaussians(
                means, scales, quats, colors_t, opacities, weights,
            )
            if means.shape[0] > int(MAX_GAUSSIANS_PER_RUN):
                idx = torch.from_numpy(
                    np.random.default_rng(42).choice(means.shape[0], int(MAX_GAUSSIANS_PER_RUN), replace=False)
                ).long()
                means, scales, quats, colors_t, opacities = (
                    means[idx], scales[idx], quats[idx], colors_t[idx], opacities[idx]
                )
            ply_out = str(out_subdir / 'gaussians.ply')
            save_gs_ply_aes(ply_out, means, scales, quats, colors_t, opacities)
            gs_rrd = build_gaussians_rrd(
                output_subdir=out_subdir,
                means=means.float().numpy(),
                colors_np=colors_t.float().numpy(),
                opacities=opacities.float().numpy(),
                scales=scales.float().numpy(),
            )

        del predictions, glb_preds, imgs
        free_cuda()

    return out_subdir, f'{S} views in {infer_time:.2f}s'

# --- Iterate ──────────────────────────────────────────────────────────
print()
log_f = open(batch_log_path, 'a', buffering=1)
n_ok = n_resume = n_reject = n_fail = 0
for scene_dir in scenes:
    scene_id = scene_dir.name
    if scene_id in already_done:
        n_resume += 1
        print(f'  [resume] {scene_id}  (already in batch_log)')
        continue
    t0 = time.time()
    try:
        out_subdir, msg = _process_one_scene(scene_dir, scene_id)
        if out_subdir is None:
            print(f'  [reject] {scene_id}  ({msg})')
            n_reject += 1
            log_f.write(json.dumps({'scene': scene_id, 'status': 'reject', 'reason': msg,
                                      'ts': datetime.now().isoformat()}) + '\n')
            continue
        dt = time.time() - t0
        print(f'  [ok]     {scene_id}  ({msg}, total {dt:.1f}s)')
        n_ok += 1
        log_f.write(json.dumps({
            'scene': scene_id,
            'status': 'ok',
            'out_dir': str(out_subdir),
            'msg': msg,
            'elapsed_s': dt,
            'ts': datetime.now().isoformat(),
        }) + '\n')
    except Exception as e:
        traceback.print_exc(limit=2)
        print(f'  [fail]   {scene_id}  → {e}')
        n_fail += 1
        log_f.write(json.dumps({'scene': scene_id, 'status': 'fail', 'error': str(e),
                                  'ts': datetime.now().isoformat()}) + '\n')
log_f.close()

print()
print('='*72)
print(f'Batch complete: {n_ok} ok / {n_resume} resumed / {n_reject} rejected / {n_fail} failed (of {len(scenes)})')
print(f'Log: {batch_log_path}')
print(f'All outputs under: {OUT_DIR}')
print('='*72)
